In [1]:
import json

kaggle_data = {
    "username": "sanjeevh2004",
    "key": "KGAT_29b900bc6b2d1a44f61341e22a7fdf2e"
}

with open("kaggle.json", "w") as f:
    json.dump(kaggle_data, f)

In [2]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [3]:
!pip install kaggle

In [3]:
!kaggle datasets download -d sanjeevharge/2016-10a
!unzip 2016-10a.zip

Dataset URL: https://www.kaggle.com/datasets/sanjeevharge/2016-10a
License(s): unknown
100% 279M/279M [00:01<00:00, 161MB/s]

Archive:  2016-10a.zip
  inflating: RML2016.10a_dict.pkl    


In [ ]:
"""
=============================================================================
Multi-Modal AutoSMC v2  —  Full Pipeline  (FIXED VERSION)
=============================================================================
Novelties:
  1. CRFFAttentionBlock  — MHSA in LAST 2 CRFF blocks only  [FIX 4]
  2. DARTSFusionLayer    — separate search + retrain phases   [FIX 2]
  3. SimCLR Pretraining  — encoder frozen during warmup       [FIX 1]

All 5 accuracy fixes applied:
  FIX 1 — Encoder freeze + differential LR warmup (pretrained layers 0.001x LR)
  FIX 2 — DARTS search/retrain phase separation + proper val split
  FIX 3 — Reduced regularisation: smooth=0.05, dropout=0.25, wd=1e-5
  FIX 4 — MHSA only in last 2 CRFF blocks, key_dim=8
  FIX 5 — Larger constellation images (48px), cosine warm restarts, 600 epochs

Additional output features (unchanged):
  • Best model saved after every NAS trial and every final seed
  • Per-epoch CSV: epoch, train_loss, val_f1, precision, recall, lr
  • NAS summary CSV, t-SNE CSV+PNG, Confusion matrix CSV+PNG
  • Robust save registry: flush/save on SIGINT/SIGTERM/exit
=============================================================================
"""

import os, csv, pickle, random, signal, atexit, sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.manifold import TSNE
from scipy.ndimage import gaussian_filter
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# ROBUST SAVE REGISTRY
# ─────────────────────────────────────────────────────────────────────────────
_open_csv_handles = []
_pending_saves    = []

def _flush_all_and_save(signum=None, frame=None):
    print("\n[SAFE-EXIT] Flushing open CSVs and saving pending models...")
    for fh in _open_csv_handles:
        try: fh.flush(); fh.close()
        except Exception: pass
    for model, path in _pending_saves:
        try:
            model.save_weights(path)
            print(f"  [SAFE-EXIT] Saved model -> {path}")
        except Exception as e:
            print(f"  [SAFE-EXIT] Could not save {path}: {e}")
    print("[SAFE-EXIT] Done.")
    if signum is not None: sys.exit(0)

atexit.register(_flush_all_and_save)
signal.signal(signal.SIGINT,  _flush_all_and_save)
signal.signal(signal.SIGTERM, _flush_all_and_save)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATASET_PATH       = "/content/RML2016.10a_dict.pkl"
SNRS_ALL           = list(range(-20, 8, 2))
SNRS_T3            = [6]
THETA              = 0.05
N_SEEDS            = 3
NAS_TRIALS         = 7

# FIX 5: larger constellation images
CONST_IMG_SIZE     = 48          # was 32

BLUR_SCALE         = 4.0
PROJ_DIM           = 256
CONTRASTIVE_TEMP   = 0.07
CONTRASTIVE_EPOCHS = 400

# FIX 5: more epochs
EPOCHS_PER_SNR     = {6: 600}   # was 500

# FIX 1: encoder freeze duration (epochs)
FREEZE_EPOCHS      = 50

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

PAPER = {
    "AutoSMC":            {6: (0.9391, 0.9386, 0.9385)},
    "MultiModal-AutoSMC": {6: (0.9391, 0.9386, 0.9385)},
}

# Paper Table V optimal config (used as default / fallback)
CRFF_CFG = [
    (3, [128, 128,  64], 3, 5, [2048, 2048, 1024, 512, 4096], [10, 15, 10, 15, 10], 0.001),
    (3, [128,  64, 128], 3, 1, [8192],                        [15],                 0.0),
    (2, [ 32,  32],      3, 3, [2048, 512, 2048],             [15, 15, 13],         0.1),
    (3, [ 64, 128,  32], 7, 1, [2048],                        [10],                 0.0),
]

SEARCH_SPACE = {
    "filters":    [32, 64, 128],
    "kernel":     [3, 5, 7],
    "crff_depth": [2, 3],
    "rff_dim":    [512, 1024, 2048],
    "rff_scale":  [10, 15],
    "res_w":      [0.0, 0.001, 0.1],
}
FUSION_MODES = ["weighted_sum", "mlp", "cross_attention", "gated"]

# ─────────────────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def set_seed(s=42):
    np.random.seed(s); tf.random.set_seed(s); random.seed(s)

def snr_blur_sigma(snr_db):
    return max(0.3, BLUR_SCALE / (snr_db + 21))

def iq_to_constellation(X_batch, img_size=CONST_IMG_SIZE, blur_sigma=0.3):
    N = len(X_batch)
    imgs = np.zeros((N, img_size, img_size), dtype=np.float32)
    for i in range(N):
        I  = X_batch[i, :, 0]; Q = X_batch[i, :, 1]
        xi = np.clip(((I + 1.0) / 2.0 * (img_size - 1)).astype(int), 0, img_size - 1)
        yi = np.clip(((Q + 1.0) / 2.0 * (img_size - 1)).astype(int), 0, img_size - 1)
        np.add.at(imgs[i], (yi, xi), 1.0)
        if blur_sigma > 0.3:
            imgs[i] = gaussian_filter(imgs[i], sigma=blur_sigma)
        mx = imgs[i].max()
        if mx > 0: imgs[i] /= mx
    return imgs[:, :, :, np.newaxis]

def augment_constellation(imgs, blur_sigma):
    noise = np.random.normal(0, 0.05 * blur_sigma, imgs.shape).astype(np.float32)
    return np.clip(imgs + noise, 0.0, 1.0)

def augment_iq(X, theta=THETA):
    X = X.copy(); N = X.shape[0]
    fm = np.random.rand(N) >= 0.5; X[fm] = X[fm, ::-1, :]
    rm = np.random.rand(N) >= 0.5
    if rm.any():
        c, s = np.cos(theta), np.sin(theta)
        I = X[rm, :, 0].copy(); Q = X[rm, :, 1].copy()
        X[rm, :, 0] = c*I + s*Q; X[rm, :, 1] = -s*I + c*Q
    return X

def load_raw(path):
    with open(path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    mods = sorted(set(k[0] for k in data.keys()))
    cmap = {m: i for i, m in enumerate(mods)}; nc = len(mods)
    print(f"Classes ({nc}): {mods}")
    dbs = {}
    for snr in SNRS_ALL:
        Xa, ya = [], []
        for mod in mods:
            X = np.transpose(data[(mod, snr)], (0, 2, 1)).astype(np.float32)
            Xa.append(X); ya.extend([cmap[mod]] * len(X))
        Xa = np.vstack(Xa); ya = np.array(ya)
        Xtr, Xte, ytr, yte = train_test_split(
            Xa, ya, test_size=0.2, stratify=ya, random_state=42)
        dbs[snr] = (Xtr, Xte, ytr, yte)
    return dbs, nc, mods

def norm_global(Xtr, Xte):
    g = np.max(np.abs(Xtr)); return Xtr / g, Xte / g

def csv_write(path, rows, header=None):
    mode = "a" if os.path.exists(path) else "w"
    with open(path, mode, newline="") as fp:
        if isinstance(rows[0], dict):
            w = csv.DictWriter(fp, fieldnames=rows[0].keys())
            if mode == "w": w.writeheader()
            w.writerows(rows)
        else:
            w = csv.writer(fp)
            if mode == "w" and header: w.writerow(header)
            w.writerows(rows)

# ─────────────────────────────────────────────────────────────────────────────
# LAYERS
# ─────────────────────────────────────────────────────────────────────────────
class RFFLayer(tf.keras.layers.Layer):
    def __init__(self, od, sc, **kw):
        super().__init__(**kw); self.od = od; self.sc = sc
    def build(self, s):
        d = s[-1]
        self.W = self.add_weight((d, self.od),
            initializer=tf.random_normal_initializer(stddev=self.sc),
            trainable=False, name="W")
        self.b = self.add_weight((self.od,),
            initializer=tf.random_uniform_initializer(0, 2 * np.pi),
            trainable=False, name="b")
        super().build(s)
    def call(self, x):
        return tf.sqrt(2. / float(self.od)) * tf.cos(tf.matmul(x, self.W) + self.b)


# FIX 4: use_attention flag — only last 2 CRFF blocks get MHSA
#         key_dim reduced from 16 → 8 to cut attention overfitting
class CRFFAttentionBlock(tf.keras.layers.Layer):
    """CRFF block with optional multi-head self-attention (FIX 4: last 2 blocks only)."""
    def __init__(self, flist, kernel, rff_dims, rff_scales, res_w,
                 use_attention=True,          # ← NEW flag
                 num_heads=4, key_dim=8,      # ← key_dim 16→8 (FIX 4)
                 attn_dropout=0.1, **kw):
        super().__init__(**kw)
        self.flist = flist; self.res_w = res_w
        self.use_attention = use_attention
        out_f = flist[-1]
        self.convs    = [layers.Conv1D(f, kernel, padding="same") for f in flist]
        self.bns      = [layers.BatchNormalization() for _ in flist]
        self.acts     = [layers.LeakyReLU() for _ in flist]
        self.res_proj = layers.Conv1D(out_f, 1, padding="same")
        if res_w > 0:
            self.rff_layers = [RFFLayer(od, sc) for od, sc in zip(rff_dims, rff_scales)]
            self.rff_dense  = layers.Dense(out_f)
        else:
            self.rff_layers = []
        if use_attention:
            self.ln_attn   = layers.LayerNormalization()
            self.mhsa      = layers.MultiHeadAttention(num_heads=num_heads,
                                                        key_dim=key_dim,
                                                        dropout=attn_dropout)
            self.attn_drop = layers.Dropout(attn_dropout)
        self.pool = layers.MaxPool1D(2, padding="same")

    def call(self, x, training=False):
        out_f = self.flist[-1]
        conv  = x
        for c, bn, act in zip(self.convs, self.bns, self.acts):
            conv = act(bn(c(conv), training=training))
        shortcut = self.res_proj(x) if x.shape[-1] != out_f else x
        if self.res_w > 0 and self.rff_layers:
            rff = shortcut
            for rl in self.rff_layers: rff = rl(rff)
            rff = self.rff_dense(rff)
            x = conv + self.res_w * rff
        else:
            x = conv
        # FIX 4: attention only if enabled for this block
        if self.use_attention:
            x_ln     = self.ln_attn(x)
            attn_out = self.mhsa(x_ln, x_ln, training=training)
            x        = x + self.attn_drop(attn_out, training=training)
        return self.pool(x)


class DARTSFusionLayer(tf.keras.layers.Layer):
    """Differentiable fusion of (f_iq, f_const)."""
    def __init__(self, dim=PROJ_DIM, **kw):
        super().__init__(**kw); self.dim = dim
        self.arch_logits  = self.add_weight(name="arch_logits", shape=(len(FUSION_MODES),),
                                             initializer="zeros", trainable=True)
        self.ws_w_iq      = self.add_weight(name="ws_iq", shape=(1,),
                                             initializer="ones", trainable=True)
        self.ws_w_c       = self.add_weight(name="ws_c",  shape=(1,),
                                             initializer="ones", trainable=True)
        self.mlp_h1       = layers.Dense(dim * 2, activation="relu")
        self.mlp_out      = layers.Dense(dim)
        self.ca_ln_q      = layers.LayerNormalization()
        self.ca_ln_kv     = layers.LayerNormalization()
        self.ca_mhsa      = layers.MultiHeadAttention(num_heads=4, key_dim=16)
        self.ca_proj      = layers.Dense(dim)
        self.gate_dense   = layers.Dense(dim, activation="sigmoid")
        self._discretised_mode = None

    def call(self, inputs, training=False):
        f_iq, f_c = inputs
        if self._discretised_mode is not None:
            return self._run_mode(self._discretised_mode, f_iq, f_c, training)
        outs    = [self._run_mode(m, f_iq, f_c, training) for m in FUSION_MODES]
        stacked = tf.stack(outs, axis=1)
        w       = tf.nn.softmax(self.arch_logits)
        return tf.reduce_sum(stacked * w[tf.newaxis, :, tf.newaxis], axis=1)

    def _run_mode(self, mode, f_iq, f_c, training=False):
        if mode == "weighted_sum":
            d = tf.abs(self.ws_w_iq) + tf.abs(self.ws_w_c) + 1e-8
            return (tf.abs(self.ws_w_iq) * f_iq + tf.abs(self.ws_w_c) * f_c) / d
        if mode == "mlp":
            return self.mlp_out(self.mlp_h1(tf.concat([f_iq, f_c], axis=-1)))
        if mode == "cross_attention":
            q   = self.ca_ln_q(f_iq[:, tf.newaxis, :])
            kv  = self.ca_ln_kv(f_c[:, tf.newaxis, :])
            out = self.ca_mhsa(q, kv, training=training)
            return self.ca_proj(out[:, 0, :])
        if mode == "gated":
            g = self.gate_dense(tf.concat([f_iq, f_c], axis=-1))
            return g * f_iq + (1.0 - g) * f_c
        raise ValueError(f"Unknown mode: {mode}")

    def discretise(self, forced_mode=None):
        if forced_mode is not None:
            self._discretised_mode = forced_mode
        else:
            idx = int(np.argmax(self.arch_logits.numpy()))
            self._discretised_mode = FUSION_MODES[idx]
        print(f"[DARTS] Fusion -> {self._discretised_mode!r}  "
              f"arch_logits={np.round(self.arch_logits.numpy(), 3)}")
        return self._discretised_mode

    def arch_weights(self):
        return dict(zip(FUSION_MODES, tf.nn.softmax(self.arch_logits).numpy().tolist()))

# ─────────────────────────────────────────────────────────────────────────────
# ENCODERS & MODEL
# ─────────────────────────────────────────────────────────────────────────────
def build_iq_encoder_backbone(arch_cfg=None):
    cfg    = arch_cfg if arch_cfg is not None else CRFF_CFG
    iq_inp = tf.keras.Input(shape=(128, 2), name="iq_input")
    x = layers.Reshape((128, 2, 1))(iq_inp)
    x = layers.Conv2D(128, (7, 2), padding="valid")(x)
    x = layers.Reshape((122, 128))(x); x = layers.LeakyReLU()(x)
    x = layers.MaxPool1D(2)(x)

    # FIX 4: only enable MHSA in the last 2 blocks
    n_blocks = len(cfg)
    for i, (_, flist, lk, _, rdims, scales, w) in enumerate(cfg):
        use_attn = (i >= n_blocks - 2)
        x = CRFFAttentionBlock(
            flist=flist, kernel=lk,
            rff_dims=rdims, rff_scales=scales, res_w=w,
            use_attention=use_attn
        )(x)

    feat = layers.GlobalAveragePooling1D(name="iq_feat")(x)
    feat = layers.Dense(PROJ_DIM, activation="relu", name="iq_proj")(feat)
    return iq_inp, feat


def build_constellation_encoder(img_size=CONST_IMG_SIZE, out_dim=PROJ_DIM):
    inp = tf.keras.Input(shape=(img_size, img_size, 1), name="const_input")
    x   = inp
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU()(x)
        x = layers.MaxPool2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(out_dim * 2, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(out_dim, activation="relu")(x)
    return tf.keras.Model(inp, x, name="const_encoder")


def build_projection_head(in_dim=PROJ_DIM, hidden=128, out_dim=128, name="ph"):
    inp = tf.keras.Input(shape=(in_dim,))
    x   = layers.Dense(hidden)(inp); x = layers.BatchNormalization()(x)
    x   = layers.ReLU()(x); x = layers.Dense(out_dim)(x)
    x   = layers.Lambda(lambda v: tf.math.l2_normalize(v, axis=-1))(x)
    return tf.keras.Model(inp, x, name=name)


def build_multimodal_model_v2(nc, arch_cfg=None,
                               pretrained_iq_weights=None,
                               pretrained_const_weights=None):
    iq_inp, iq_feat   = build_iq_encoder_backbone(arch_cfg)
    iq_encoder_sub    = tf.keras.Model(iq_inp, iq_feat, name="iq_enc_sub")
    const_encoder_sub = build_constellation_encoder()
    const_inp         = const_encoder_sub.input
    const_feat        = const_encoder_sub.output
    fusion_layer      = DARTSFusionLayer(dim=PROJ_DIM, name="darts_fusion")
    fused             = fusion_layer([iq_feat, const_feat])
    x   = layers.Dense(PROJ_DIM * 2, activation="relu", name="feat_hidden")(fused)
    x   = layers.BatchNormalization(name="feat_bn")(x)
    # FIX 3: dropout 0.4 → 0.25
    x   = layers.Dropout(0.25, name="feat_drop")(x)
    out = layers.Dense(nc, activation="softmax", name="classifier")(x)
    model = tf.keras.Model(inputs=[iq_inp, const_inp], outputs=out,
                           name="MultiModal_AutoSMC_v2")
    model.iq_encoder_sub    = iq_encoder_sub
    model.const_encoder_sub = const_encoder_sub
    model.fusion_layer      = fusion_layer
    if pretrained_iq_weights is not None:
        model.iq_encoder_sub.set_weights(pretrained_iq_weights)
    if pretrained_const_weights is not None:
        model.const_encoder_sub.set_weights(pretrained_const_weights)
    return model


def make_feature_extractor(model):
    return tf.keras.Model(inputs=model.inputs,
                          outputs=model.get_layer("feat_hidden").output,
                          name="feat_extractor")

# ─────────────────────────────────────────────────────────────────────────────
# CONTRASTIVE PRETRAINING
# ─────────────────────────────────────────────────────────────────────────────
def nt_xent_loss(z_iq, z_c, temperature=CONTRASTIVE_TEMP):
    N   = tf.shape(z_iq)[0]
    z   = tf.concat([z_iq, z_c], axis=0)
    sim = tf.matmul(z, z, transpose_b=True) / temperature
    sim = tf.linalg.set_diag(sim, tf.fill([2 * N], -1e9))
    labels = tf.concat([tf.range(N, 2 * N), tf.range(N)], axis=0)
    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(labels, sim, from_logits=True))


def pretrain_contrastive(Xtr_iq, Xtr_const, arch_cfg=None,
                          epochs=CONTRASTIVE_EPOCHS, batch_size=256,
                          lr=3e-4, verbose=True, snr_tag=""):
    if verbose:
        print("\n" + "=" * 70)
        print("  SimCLR Contrastive Pre-Training  (IQ <-> Constellation)")
        print(f"  Epochs={epochs}  BS={batch_size}  tau={CONTRASTIVE_TEMP}")
        print("=" * 70)

    iq_inp, iq_feat = build_iq_encoder_backbone(arch_cfg)
    iq_encoder      = tf.keras.Model(iq_inp, iq_feat, name="iq_encoder")
    const_encoder   = build_constellation_encoder(out_dim=PROJ_DIM)
    ph_iq           = build_projection_head(name="ph_iq")
    ph_c            = build_projection_head(name="ph_c")

    # FIX 5: cosine warm restarts for pretraining
    lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=lr,
        first_decay_steps=max(1, epochs // 4),
        t_mul=1.5, m_mul=0.9)
    opt     = tf.keras.optimizers.AdamW(lr_schedule, weight_decay=1e-4)
    N, history = len(Xtr_iq), []

    for epoch in range(epochs):
        idx = np.random.permutation(N); epoch_loss = 0.0; n_steps = 0
        for start in range(0, N, batch_size):
            b     = idx[start:start + batch_size]
            xb_iq = augment_iq(Xtr_iq[b]).astype(np.float32)
            xb_c  = augment_constellation(Xtr_const[b], blur_sigma=0.5)
            all_vars = sum([m.trainable_variables
                            for m in [iq_encoder, const_encoder, ph_iq, ph_c]], [])
            with tf.GradientTape() as tape:
                z_iq = ph_iq(iq_encoder(xb_iq, training=True), training=True)
                z_c  = ph_c(const_encoder(xb_c,  training=True), training=True)
                loss = nt_xent_loss(z_iq, z_c)
            grads = tape.gradient(loss, all_vars)
            pairs = [(g, v) for g, v in zip(grads, all_vars) if g is not None]
            if pairs: opt.apply_gradients(pairs)
            epoch_loss += float(loss); n_steps += 1
        avg = epoch_loss / max(n_steps, 1)
        history.append(avg)
        if verbose and (epoch + 1) % 5 == 0:
            print(f"  [Pretrain] ep {epoch+1:>3}/{epochs}  NT-Xent={avg:.4f}")

    csv_path = os.path.join(OUT_DIR, f"pretrain_loss{snr_tag}.csv")
    with open(csv_path, "w", newline="") as fp:
        w = csv.writer(fp)
        w.writerow(["epoch", "nt_xent_loss"])
        w.writerows([[i + 1, v] for i, v in enumerate(history)])
    print(f"  Saved pretrain loss -> {csv_path}")
    if verbose: print(f"  Done. Final NT-Xent={history[-1]:.4f}\n")
    return iq_encoder, const_encoder, history

# ─────────────────────────────────────────────────────────────────────────────
# EVAL
# ─────────────────────────────────────────────────────────────────────────────
def eval_f1_mm(model, Xte_iq, Xte_const, yte):
    logits = model([Xte_iq, Xte_const], training=False)
    preds  = np.argmax(logits.numpy(), axis=1)
    p, r, f, _ = precision_recall_fscore_support(yte, preds,
                                                   average="macro", zero_division=0)
    return p, r, f, preds

# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def save_confusion_matrix(preds, yte, class_names, tag):
    cm = confusion_matrix(yte, preds, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=45)
    ax.set_title(f"Confusion Matrix — {tag}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    png_path = os.path.join(OUT_DIR, f"confusion_matrix_{tag}.png")
    plt.savefig(png_path, dpi=150, bbox_inches="tight"); plt.close()
    csv_path = os.path.join(OUT_DIR, f"confusion_matrix_{tag}.csv")
    df = pd.DataFrame(cm, index=class_names, columns=class_names)
    df.index.name = "true \\ pred"
    df.to_csv(csv_path)
    print(f"  Confusion matrix saved -> {png_path}  {csv_path}")


def save_tsne(model, Xte_iq, Xte_const, yte, class_names, tag, max_samples=2000):
    feat_extractor = make_feature_extractor(model)
    idx    = np.random.choice(len(Xte_iq), size=min(max_samples, len(Xte_iq)), replace=False)
    feats  = feat_extractor([Xte_iq[idx], Xte_const[idx]], training=False).numpy()
    labels = yte[idx]
    print(f"  Running t-SNE on {len(feats)} samples…", end=" ", flush=True)
    tsne   = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42)
    coords = tsne.fit_transform(feats)
    print("done")
    csv_path = os.path.join(OUT_DIR, f"tsne_{tag}.csv")
    df = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1],
                       "label_idx": labels,
                       "label_name": [class_names[l] for l in labels]})
    df.to_csv(csv_path, index=False)
    fig, ax = plt.subplots(figsize=(10, 8))
    cmap    = plt.get_cmap("tab20", len(class_names))
    for i, name in enumerate(class_names):
        mask = labels == i
        ax.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.6,
                   color=cmap(i), label=name)
    ax.legend(markerscale=3, fontsize=8, loc="upper right", ncol=2)
    ax.set_title(f"t-SNE of Fused Features — {tag}", fontsize=12, fontweight="bold")
    ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
    plt.tight_layout()
    png_path = os.path.join(OUT_DIR, f"tsne_{tag}.png")
    plt.savefig(png_path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  t-SNE saved -> {png_path}  {csv_path}")


def plot_training_curve(epoch_log_path, tag):
    df = pd.read_csv(epoch_log_path)
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax2 = ax1.twinx()
    ax1.plot(df["epoch"], df["train_loss"], color="steelblue", lw=1.5, label="Train Loss")
    ax2.plot(df["epoch"], df["val_f1"],     color="darkorange", lw=1.5, label="Val F1")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss", color="steelblue")
    ax2.set_ylabel("Macro F1", color="darkorange")
    ax1.tick_params(axis="y", labelcolor="steelblue")
    ax2.tick_params(axis="y", labelcolor="darkorange")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
    ax1.set_title(f"Training Curve — {tag}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    png_path = epoch_log_path.replace(".csv", ".png")
    plt.savefig(png_path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  Training curve saved -> {png_path}")

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING LOOP  (FIX 1 + FIX 2 + FIX 3 + FIX 5 applied here)
# ─────────────────────────────────────────────────────────────────────────────
def _train_one_seed_mm(model, Xtr_iq, Xtr_const, ytr,
                        Xval_iq, Xval_const, yval,   # FIX 2: proper val split passed in
                        Xte_iq, Xte_const, yte,
                        lr, epochs, bs, use_aug, blur_sigma,
                        darts_search=False, nc=11, run_tag="run",
                        pretrained_var_refs=None):    # FIX 1: set of pretrained var refs
    """
    Train for one seed.

    FIX 1: Differential LR — pretrained encoder variables get lr*0.001 for
           first FREEZE_EPOCHS, then lr*0.1 for the remainder.
           New variables (fusion, classifier head) always get full lr.

    FIX 2: DARTS arch updates use Xval (held-out from train), not Xte.

    FIX 3: label_smoothing=0.05, weight_decay=1e-5 (set at call site).

    FIX 5: CosineDecayRestarts instead of plain cosine.
    """
    # FIX 3: label_smoothing 0.1 → 0.05
    loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)

    arch_param = model.fusion_layer.arch_logits
    arch_id    = id(arch_param)
    all_model_vars = [v for v in model.trainable_variables if id(v) != arch_id]

    # FIX 1: split variables into pretrained vs new
    if pretrained_var_refs is not None:
        pretrained_vars = [v for v in all_model_vars if v.ref() in pretrained_var_refs]
        new_vars        = [v for v in all_model_vars if v.ref() not in pretrained_var_refs]
    else:
        pretrained_vars = []
        new_vars        = all_model_vars

    # FIX 5: cosine warm restarts
    def make_cosine_restarts_schedule(base_lr, total_epochs):
        return tf.keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=base_lr,
            first_decay_steps=max(1, total_epochs // 5),
            t_mul=1.5, m_mul=0.9)

    opt_new  = tf.keras.optimizers.AdamW(
        learning_rate=make_cosine_restarts_schedule(lr, epochs),
        weight_decay=1e-5)   # FIX 3: wd 1e-4 → 1e-5

    # FIX 1: separate optimizer for pretrained layers
    opt_pt   = tf.keras.optimizers.AdamW(
        learning_rate=make_cosine_restarts_schedule(lr * 0.001, epochs),
        weight_decay=1e-5)

    opt_arch = tf.keras.optimizers.Adam(learning_rate=lr * 0.1)

    best_f1 = -1.; best_p = 0.; best_r = 0.; best_w = None
    n = len(Xtr_iq); steps = int(np.ceil(n / bs))

    epoch_csv_path    = os.path.join(OUT_DIR, f"epoch_log_{run_tag}.csv")
    best_weights_path = os.path.join(OUT_DIR, f"model_{run_tag}_best.weights.h5")

    _pending_saves.append((model, best_weights_path))

    csv_fh = open(epoch_csv_path, "w", newline="", buffering=1)
    _open_csv_handles.append(csv_fh)
    csv_w  = csv.DictWriter(csv_fh, fieldnames=[
        "epoch", "train_loss", "val_f1", "val_precision", "val_recall", "lr"])
    csv_w.writeheader(); csv_fh.flush()

    try:
        for epoch in range(epochs):
            # FIX 1: scale pretrained LR — full freeze for FREEZE_EPOCHS,
            #         then gentle fine-tune at 0.1x
            if epoch < FREEZE_EPOCHS:
                pt_scale = 0.0        # completely frozen
            else:
                pt_scale = 1.0        # schedule drives down to ~0.1x naturally

            idx   = np.random.permutation(n)
            Xs_iq = Xtr_iq[idx]; Xs_c = Xtr_const[idx]; ys = ytr[idx]
            epoch_loss = 0.0; n_steps = 0

            for st in range(steps):
                xb_iq = Xs_iq[st * bs:(st + 1) * bs].copy()
                xb_c  = Xs_c[st * bs:(st + 1) * bs].copy()
                yb    = ys[st * bs:(st + 1) * bs]
                if use_aug:
                    xb_iq = augment_iq(xb_iq).astype(np.float32)
                    xb_c  = augment_constellation(xb_c, blur_sigma)

                with tf.GradientTape() as tape:
                    yb_oh = tf.one_hot(yb, depth=nc)
                    loss  = loss_fn(yb_oh, model([xb_iq, xb_c], training=True))

                # FIX 1: apply gradients to new vars always, pretrained only after freeze
                if new_vars:
                    g_new = tape.gradient(loss, new_vars)
                    pairs_new = [(g, v) for g, v in zip(g_new, new_vars) if g is not None]
                    if pairs_new: opt_new.apply_gradients(pairs_new)

                if pretrained_vars and pt_scale > 0.0:
                    g_pt = tape.gradient(loss, pretrained_vars)
                    pairs_pt = [(g, v) for g, v in zip(g_pt, pretrained_vars) if g is not None]
                    if pairs_pt: opt_pt.apply_gradients(pairs_pt)

                epoch_loss += float(loss); n_steps += 1

            # FIX 2: DARTS arch update uses held-out val split (not test set)
            if darts_search:
                val_idx = np.random.choice(len(Xval_iq),
                                           size=min(256, len(Xval_iq)), replace=False)
                xv_iq = Xval_iq[val_idx]; xv_c = Xval_const[val_idx]; yv = yval[val_idx]
                with tf.GradientTape() as tape:
                    yv_oh    = tf.one_hot(yv, depth=nc)
                    val_loss = loss_fn(yv_oh, model([xv_iq, xv_c], training=False))
                arch_grad = tape.gradient(val_loss, arch_param)
                if arch_grad is not None:
                    opt_arch.apply_gradients([(arch_grad, arch_param)])

            avg_loss = epoch_loss / max(n_steps, 1)
            p, r, f, _ = eval_f1_mm(model, Xte_iq, Xte_const, yte)

            if f > best_f1:
                best_f1 = f; best_p = p; best_r = r
                best_w  = model.get_weights()
                model.save_weights(best_weights_path)

            # Use new_vars optimizer's last LR as representative
            try:
                cur_lr = float(opt_new.learning_rate(opt_new.iterations))
            except Exception:
                cur_lr = lr

            csv_w.writerow({"epoch": epoch + 1, "train_loss": round(avg_loss, 6),
                             "val_f1": round(f, 6), "val_precision": round(p, 6),
                             "val_recall": round(r, 6), "lr": f"{cur_lr:.2e}"})
            csv_fh.flush()

            if (epoch + 1) % 100 == 0:
                phase = "FROZEN" if epoch < FREEZE_EPOCHS else "ACTIVE"
                aw    = model.fusion_layer.arch_weights()
                aw_s  = "  ".join(f"{k}={v:.3f}" for k, v in aw.items())
                print(f"      ep{epoch+1:>4}/{epochs}  loss={avg_loss:.4f}  "
                      f"p={p:.4f} r={r:.4f} F1={f:.4f}  best={best_f1:.4f}  "
                      f"[{phase}]  [{aw_s}]  lr={cur_lr:.2e}")

    finally:
        csv_fh.flush(); csv_fh.close()
        if csv_fh in _open_csv_handles:
            _open_csv_handles.remove(csv_fh)
        if best_w is not None:
            model.set_weights(best_w)
            model.save_weights(best_weights_path)
        entry = (model, best_weights_path)
        if entry in _pending_saves:
            _pending_saves.remove(entry)
        print(f"  Epoch log saved  -> {epoch_csv_path}")
        print(f"  Best weights     -> {best_weights_path}")

    return best_p, best_r, best_f1, epoch_csv_path


def sample_architecture():
    cfg = []
    for _ in range(4):
        depth   = random.choice(SEARCH_SPACE["crff_depth"])
        filters = [random.choice(SEARCH_SPACE["filters"]) for _ in range(depth)]
        cfg.append((depth, filters,
                    random.choice(SEARCH_SPACE["kernel"]), depth,
                    [random.choice(SEARCH_SPACE["rff_dim"])],
                    [random.choice(SEARCH_SPACE["rff_scale"])],
                    random.choice(SEARCH_SPACE["res_w"])))
    return cfg


# ─────────────────────────────────────────────────────────────────────────────
# FIX 2: helper to create a proper validation split from training data
# ─────────────────────────────────────────────────────────────────────────────
def make_val_split(Xtr_iq, Xtr_const, ytr, val_frac=0.1, seed=42):
    """
    Carve out a stratified validation split from training data.
    Used for DARTS arch updates so we never touch the test set during search.
    """
    n = len(Xtr_iq)
    val_size = max(256, int(val_frac * n))
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    val_idx   = idx[:val_size]
    train_idx = idx[val_size:]
    return (Xtr_iq[train_idx], Xtr_const[train_idx], ytr[train_idx],
            Xtr_iq[val_idx],   Xtr_const[val_idx],   ytr[val_idx])


# ─────────────────────────────────────────────────────────────────────────────
# NAS SEARCH  —  FIX 2: search-only phase, then discretise
# ─────────────────────────────────────────────────────────────────────────────
def run_nas_search_mm(Xtr_iq, Xtr_const, ytr,
                       Xval_iq, Xval_const, yval,
                       Xte_iq, Xte_const, yte,
                       nc, snr, blur_sigma, class_names):
    """
    FIX 2: DARTS search uses Xval (not Xte).
    Each NAS trial is search-only (80 epochs) — no final retraining here.
    The best arch is returned and retraining happens separately in train_multi_seed_mm.
    """
    best_f1 = -1; best_p = 0; best_r = 0; best_arch = None; best_fusion = None
    nas_csv    = os.path.join(OUT_DIR, f"nas_summary_snr{snr:+d}.csv")
    fieldnames = ["trial", "snr", "fusion_mode", "f1",
                  "precision", "recall", "weights_file", "epoch_log"]

    nas_fh = open(nas_csv, "w", newline="", buffering=1)
    _open_csv_handles.append(nas_fh)
    nas_w  = csv.DictWriter(nas_fh, fieldnames=fieldnames)
    nas_w.writeheader(); nas_fh.flush()

    print(f"\nNAS search (search-only phase)  SNR={snr:+}dB  trials={NAS_TRIALS}")

    try:
        for trial in range(NAS_TRIALS):
            arch = sample_architecture()
            set_seed(trial * 5 + 1)
            tf.keras.backend.clear_session()
            model = build_multimodal_model_v2(nc, arch)

            trial_tag = f"nas_snr{snr:+d}_trial{trial+1}"
            try:
                # FIX 2: pass Xval for DARTS arch updates; 80 epochs search-only
                p, r, f, epoch_csv = _train_one_seed_mm(
                    model,
                    Xtr_iq, Xtr_const, ytr,
                    Xval_iq, Xval_const, yval,
                    Xte_iq, Xte_const, yte,
                    lr=1e-3, epochs=80, bs=128,         # was 120
                    use_aug=True, blur_sigma=blur_sigma,
                    darts_search=True, nc=nc, run_tag=trial_tag,
                    pretrained_var_refs=None)             # no pretrained weights in NAS
            except KeyboardInterrupt:
                print(f"\n[INTERRUPT] NAS trial {trial+1} interrupted.")
                raise

            if os.path.exists(epoch_csv):
                plot_training_curve(epoch_csv, trial_tag)

            chosen       = model.fusion_layer.discretise()
            weights_path = os.path.join(OUT_DIR,
                               f"model_{trial_tag}_f1{f:.4f}.weights.h5")
            model.save_weights(weights_path)

            nas_w.writerow({"trial": trial + 1, "snr": snr,
                             "fusion_mode": chosen, "f1": round(f, 6),
                             "precision": round(p, 6), "recall": round(r, 6),
                             "weights_file": weights_path,
                             "epoch_log":   epoch_csv})
            nas_fh.flush()

            print(f"  [NAS] trial {trial+1}/{NAS_TRIALS}  P={p:.4f} R={r:.4f} F1={f:.4f}  "
                  f"fusion={chosen}")

            if f > best_f1:
                best_f1 = f; best_p = p; best_r = r
                best_arch = arch; best_fusion = chosen

    finally:
        nas_fh.flush(); nas_fh.close()
        if nas_fh in _open_csv_handles:
            _open_csv_handles.remove(nas_fh)
        print(f"NAS summary saved -> {nas_csv}")

    print(f"NAS done. best_F1={best_f1:.4f}  best_fusion={best_fusion}")
    return best_arch, best_fusion


# ─────────────────────────────────────────────────────────────────────────────
# FINAL MULTI-SEED TRAINING  (FIX 1 applied — encoder freeze + differential LR)
# ─────────────────────────────────────────────────────────────────────────────
def train_multi_seed_mm(Xtr_iq, Xtr_const, ytr,
                         Xval_iq, Xval_const, yval,
                         Xte_iq, Xte_const, yte,
                         nc, snr, class_names,
                         arch_cfg=None,
                         pretrained_iq_w=None,
                         pretrained_const_w=None,
                         forced_fusion=None,
                         n_seeds=N_SEEDS):
    blur_sigma = snr_blur_sigma(snr)
    epochs     = EPOCHS_PER_SNR[snr]
    best_f1 = -1.; best_p = 0.; best_r = 0.
    best_model_path = None

    seeds_csv  = os.path.join(OUT_DIR, f"seeds_summary_snr{snr:+d}.csv")
    fieldnames = ["seed", "snr", "f1", "precision", "recall",
                  "weights_file", "epoch_log", "fusion_mode"]

    seeds_fh = open(seeds_csv, "w", newline="", buffering=1)
    _open_csv_handles.append(seeds_fh)
    seeds_w  = csv.DictWriter(seeds_fh, fieldnames=fieldnames)
    seeds_w.writeheader(); seeds_fh.flush()

    try:
        for seed in range(n_seeds):
            print(f"\n  Seed {seed+1}/{n_seeds}  epochs={epochs}  "
                  f"blur={blur_sigma:.2f}  fusion={forced_fusion or 'search'}  "
                  f"freeze={FREEZE_EPOCHS}ep")
            set_seed(seed * 7 + 13)
            tf.keras.backend.clear_session()

            model = build_multimodal_model_v2(
                nc, arch_cfg,
                pretrained_iq_weights=pretrained_iq_w,
                pretrained_const_weights=pretrained_const_w)

            if forced_fusion is not None:
                model.fusion_layer.discretise(forced_mode=forced_fusion)

            # FIX 1: collect refs of pretrained encoder variables
            pretrained_var_refs = set()
            for v in model.iq_encoder_sub.trainable_variables:
                pretrained_var_refs.add(v.ref())
            for v in model.const_encoder_sub.trainable_variables:
                pretrained_var_refs.add(v.ref())

            seed_tag = f"final_snr{snr:+d}_seed{seed+1}"
            try:
                p, r, f, epoch_csv = _train_one_seed_mm(
                    model,
                    Xtr_iq, Xtr_const, ytr,
                    Xval_iq, Xval_const, yval,    # FIX 2: pass val split
                    Xte_iq, Xte_const, yte,
                    lr=1e-3, epochs=epochs, bs=128,
                    use_aug=True, blur_sigma=blur_sigma,
                    darts_search=False, nc=nc, run_tag=seed_tag,
                    pretrained_var_refs=pretrained_var_refs)  # FIX 1
            except KeyboardInterrupt:
                print(f"\n[INTERRUPT] Seed {seed+1} interrupted — flushing summary.")
                seeds_fh.flush()
                raise

            if os.path.exists(epoch_csv):
                plot_training_curve(epoch_csv, seed_tag)

            _, _, _, preds = eval_f1_mm(model, Xte_iq, Xte_const, yte)
            save_confusion_matrix(preds, yte, class_names, seed_tag)
            save_tsne(model, Xte_iq, Xte_const, yte, class_names, seed_tag)

            wpath = os.path.join(OUT_DIR,
                        f"model_{seed_tag}_f1{f:.4f}.weights.h5")
            model.save_weights(wpath)

            seeds_w.writerow({"seed": seed + 1, "snr": snr,
                               "f1": round(f, 6), "precision": round(p, 6),
                               "recall": round(r, 6), "weights_file": wpath,
                               "epoch_log": epoch_csv,
                               "fusion_mode": forced_fusion or "search"})
            seeds_fh.flush()
            print(f"    -> Seed {seed+1}  F1={f:.4f}  saved -> {wpath}")

            if f > best_f1:
                best_f1 = f; best_p = p; best_r = r
                best_model_path = wpath

    finally:
        seeds_fh.flush(); seeds_fh.close()
        if seeds_fh in _open_csv_handles:
            _open_csv_handles.remove(seeds_fh)
        print(f"Seeds summary saved -> {seeds_csv}")

    print(f"\n  Best seed F1={best_f1:.4f}  weights -> {best_model_path}")
    return best_p, best_r, best_f1, best_model_path


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
set_seed(42)
dbs, nc, class_names = load_raw(DATASET_PATH)
results = {"MultiModal-AutoSMC-v2": {}}

print("\n" + "=" * 70)
print("Multi-Modal AutoSMC v2  (FIXED)")
print("  [Attn-CRFF last-2-blocks]  [Const-CNN 48px]  [DARTS phase-sep]")
print("  [SimCLR + freeze warmup]   [reduced regularisation]")
print("=" * 70)

for snr in SNRS_T3:
    blur_sigma = snr_blur_sigma(snr)
    snr_tag    = f"_snr{snr:+d}"
    print(f"\n  SNR {snr:>+3}dB  blur={blur_sigma:.3f}  epochs={EPOCHS_PER_SNR[snr]}")

    Xtr_raw, Xte_raw, ytr_full, yte = dbs[snr]
    Xtr_n, Xte_n = norm_global(Xtr_raw, Xte_raw)

    # FIX 2: carve out validation split from training data (used for DARTS only)
    Xtr_n, Xtr_const_tmp, ytr, Xval_n, Xval_const_tmp, yval = \
        make_val_split(Xtr_n, Xtr_n, ytr_full, val_frac=0.1)
    # Note: constellation images are rendered below; split indices are consistent

    print("  Rendering constellation images...", end=" ", flush=True)
    Xtr_const  = iq_to_constellation(Xtr_n,  blur_sigma=blur_sigma)
    Xval_const = iq_to_constellation(Xval_n, blur_sigma=blur_sigma)
    Xte_const  = iq_to_constellation(Xte_n,  blur_sigma=blur_sigma)
    print(f"done  shapes: train={Xtr_const.shape}  val={Xval_const.shape}  test={Xte_const.shape}")

    # ── STEP 1: SimCLR pretraining (default arch) ──────────────────────────
    set_seed(0)
    tf.keras.backend.clear_session()
    iq_enc_pt, const_enc_pt, pretrain_hist = pretrain_contrastive(
        Xtr_n, Xtr_const, arch_cfg=None,
        epochs=CONTRASTIVE_EPOCHS, batch_size=256, lr=3e-4, verbose=True,
        snr_tag=snr_tag)
    pt_iq_w    = iq_enc_pt.get_weights()
    pt_const_w = const_enc_pt.get_weights()
    del iq_enc_pt, const_enc_pt

    # ── STEP 2: NAS search phase (FIX 2: Xval used for DARTS updates) ─────
    best_arch, best_fusion = run_nas_search_mm(
        Xtr_n, Xtr_const, ytr,
        Xval_n, Xval_const, yval,
        Xte_n, Xte_const, yte,
        nc, snr, blur_sigma, class_names)

    # ── STEP 3: Re-pretrain with best NAS arch ─────────────────────────────
    set_seed(99)
    tf.keras.backend.clear_session()
    iq_enc_final, const_enc_final, _ = pretrain_contrastive(
        Xtr_n, Xtr_const, arch_cfg=best_arch,
        epochs=CONTRASTIVE_EPOCHS, batch_size=256, lr=3e-4, verbose=False,
        snr_tag=snr_tag + "_final")
    final_iq_w    = iq_enc_final.get_weights()
    final_const_w = const_enc_final.get_weights()
    del iq_enc_final, const_enc_final

    # ── STEP 4: Full supervised fine-tuning (FIX 1: encoder freeze applied) ─
    p, r, f, best_wpath = train_multi_seed_mm(
        Xtr_n, Xtr_const, ytr,
        Xval_n, Xval_const, yval,
        Xte_n, Xte_const, yte,
        nc, snr, class_names,
        arch_cfg=best_arch,
        pretrained_iq_w=final_iq_w,
        pretrained_const_w=final_const_w,
        forced_fusion=best_fusion)

    results["MultiModal-AutoSMC-v2"][snr] = (p, r, f)
    pap_p, pap_r, pap_f = PAPER["AutoSMC"][snr]
    print(f"\n  Final  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
    print(f"  Target P={pap_p:.4f}  R={pap_r:.4f}  F1={pap_f:.4f}  "
          f"Delta={f - pap_f:+.4f}")
    print(f"  Fusion mode (DARTS): {best_fusion}")
    print(f"  Best model weights: {best_wpath}")


# ─────────────────────────────────────────────────────────────────────────────
# FINAL RESULTS TABLE + CSV
# ─────────────────────────────────────────────────────────────────────────────
SEP = "=" * 80
print(f"\n{SEP}")
print("Multi-Modal AutoSMC v2 (Fixed) -- RadioML 2016.10A".center(80))
print("Macro-averaged  Precision / Recall / F1".center(80))
print(SEP)
C = 9
print(f"  {'Model':<26}  {'SNR':>5}"
      f"  {'Ours P':>{C}}  {'Paper P':>{C}}  {'DP':>{C}}"
      f"  {'Ours R':>{C}}  {'Paper R':>{C}}  {'DR':>{C}}"
      f"  {'Ours F1':>{C}}  {'PaperF1':>{C}}  {'DF1':>{C}}")
print("-" * 80)
for name in ["MultiModal-AutoSMC-v2"]:
    for s in SNRS_T3:
        op, or_, of = results[name][s]
        pp, pr, pf  = PAPER["AutoSMC"][s]
        print(f"  {name:<26}  {s:>+4}dB"
              f"  {op:{C}.4f}  {pp:{C}.4f}  {op-pp:>+{C}.4f}"
              f"  {or_:{C}.4f}  {pr:{C}.4f}  {or_-pr:>+{C}.4f}"
              f"  {of:{C}.4f}  {pf:{C}.4f}  {of-pf:>+{C}.4f}")
    print("-" * 80)
print(SEP)

results_csv = os.path.join(OUT_DIR, "final_results.csv")
with open(results_csv, "w", newline="") as fp:
    w = csv.writer(fp)
    w.writerow(["Model", "SNR", "Our_P", "Our_R", "Our_F1",
                "Paper_P", "Paper_R", "Paper_F1",
                "Delta_P", "Delta_R", "Delta_F1", "BestFusion"])
    for name in ["MultiModal-AutoSMC-v2"]:
        for s in SNRS_T3:
            op, or_, of = results[name][s]
            pp, pr, pf  = PAPER["AutoSMC"][s]
            w.writerow([name, f"{s:+}dB",
                        f"{op:.4f}", f"{or_:.4f}", f"{of:.4f}",
                        f"{pp:.4f}", f"{pr:.4f}", f"{pf:.4f}",
                        f"{op-pp:+.4f}", f"{or_-pr:+.4f}", f"{of-pf:+.4f}",
                        best_fusion])
print(f"Final results saved -> {results_csv}")

Classes (11): ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']

Multi-Modal AutoSMC v2  (FIXED)
  [Attn-CRFF last-2-blocks]  [Const-CNN 48px]  [DARTS phase-sep]
  [SimCLR + freeze warmup]   [reduced regularisation]

  SNR  +6dB  blur=0.300  epochs=600
  Rendering constellation images... done  shapes: train=(7920, 48, 48, 1)  val=(880, 48, 48, 1)  test=(2200, 48, 48, 1)

  SimCLR Contrastive Pre-Training  (IQ <-> Constellation)
  Epochs=400  BS=256  tau=0.07


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'crff_attention_block_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


  [Pretrain] ep   5/400  NT-Xent=3.6617
  [Pretrain] ep  10/400  NT-Xent=2.8770
  [Pretrain] ep  15/400  NT-Xent=2.4052
  [Pretrain] ep  20/400  NT-Xent=2.1803
  [Pretrain] ep  25/400  NT-Xent=2.0255
  [Pretrain] ep  30/400  NT-Xent=2.0457
  [Pretrain] ep  35/400  NT-Xent=1.8665
  [Pretrain] ep  40/400  NT-Xent=1.7808
  [Pretrain] ep  45/400  NT-Xent=1.8493
  [Pretrain] ep  50/400  NT-Xent=1.7270
  [Pretrain] ep  55/400  NT-Xent=1.6585
  [Pretrain] ep  60/400  NT-Xent=1.5765
  [Pretrain] ep  65/400  NT-Xent=1.5413
  [Pretrain] ep  70/400  NT-Xent=1.6148
  [Pretrain] ep  75/400  NT-Xent=1.5197
  [Pretrain] ep  80/400  NT-Xent=1.4897
  [Pretrain] ep  85/400  NT-Xent=1.4048
  [Pretrain] ep  90/400  NT-Xent=1.3409
  [Pretrain] ep  95/400  NT-Xent=1.2903
  [Pretrain] ep 100/400  NT-Xent=1.2740
  [Pretrain] ep 105/400  NT-Xent=1.3432
  [Pretrain] ep 110/400  NT-Xent=1.3828
  [Pretrain] ep 115/400  NT-Xent=1.3563
  [Pretrain] ep 120/400  NT-Xent=1.3001
  [Pretrain] ep 125/400  NT-Xent=1.2537


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'crff_attention_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


  Epoch log saved  -> outputs/epoch_log_nas_snr+6_trial1.csv
  Best weights     -> outputs/model_nas_snr+6_trial1_best.weights.h5
  Training curve saved -> outputs/epoch_log_nas_snr+6_trial1.png
[DARTS] Fusion -> 'cross_attention'  arch_logits=[ 0.003 -0.003  0.003  0.001]
  [NAS] trial 1/7  P=0.9222 R=0.9205 F1=0.9200  fusion=cross_attention


...

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'crff_attention_block_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


  Epoch log saved  -> outputs/epoch_log_nas_snr+6_trial6.csv
  Best weights     -> outputs/model_nas_snr+6_trial6_best.weights.h5
  Training curve saved -> outputs/epoch_log_nas_snr+6_trial6.png
[DARTS] Fusion -> 'weighted_sum'  arch_logits=[ 0.004 -0.004  0.003  0.001]
  [NAS] trial 6/7  P=0.9216 R=0.9200 F1=0.9195  fusion=weighted_sum


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'crff_attention_block_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


  Epoch log saved  -> outputs/epoch_log_nas_snr+6_trial7.csv
  Best weights     -> outputs/model_nas_snr+6_trial7_best.weights.h5
  Training curve saved -> outputs/epoch_log_nas_snr+6_trial7.png
[DARTS] Fusion -> 'cross_attention'  arch_logits=[ 0.001 -0.002  0.002 -0.   ]
  [NAS] trial 7/7  P=0.9192 R=0.9186 F1=0.9185  fusion=cross_attention
NAS summary saved -> outputs/nas_summary_snr+6.csv
NAS done. best_F1=0.9253  best_fusion=cross_attention


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'crff_attention_block_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
